# WORD2VEC - SKIP-GRAM (from scratch, pure NumPy)

This notebook is a line-by-line implementation of the pseudocode:

```
WORD2VEC-SKIPGRAM(Corpus, c, d, eta, T)

1. Build vocabulary V from the corpus
2. Initialize input embeddings E randomly
3. Initialize output weights W randomly

4. Generate training pairs:
   For each sentence in Corpus:
     For each word w at position i:
       For each context position j within window c:
         If i != j:
           Add pair (w, context_word)

5. For epoch = 1 to T:
     Shuffle training pairs
     For each pair (center, context):

         # Forward pass
         h = E[center]
         scores = h x W
         probabilities = Softmax(scores)

         # Compute loss
         loss = -log(probabilities[context])

         # Backpropagation
         error = probabilities
         error[context] -= 1

         gradient_W = h^T x error
         gradient_h = W x error

         # Update parameters
         W = W - eta * gradient_W
         E[center] = E[center] - eta * gradient_h

6. Return E
```

Every cell below is commented with **what** is happening and **why**, so this notebook is self-contained — you should not need to look anything up elsewhere to follow it.

### Notation used throughout (matches the pseudocode)

| Symbol | Meaning |
|---|---|
| `Corpus` | list of sentences, each sentence is a list of words (tokens) |
| `c` | window size (how many words to the left/right count as context) |
| `d` | embedding dimension (size of each word vector) |
| `eta` | learning rate (step size for gradient descent) |
| `T` | number of training epochs |
| `V` | vocabulary size (number of unique words) |
| `E` | input embedding matrix, shape `(V, d)` — one row per word |
| `W` | output weight matrix, shape `(d, V)` — scores every vocabulary word against a center word's embedding |

In [1]:
import numpy as np

## Step 1: Build vocabulary V from the corpus

Neural networks cannot operate on raw strings, so every unique word gets assigned an integer id. That id is what we actually use to index into the embedding matrix `E`.

In [2]:
def build_vocabulary(corpus):
    """
    Scans every sentence/word in the corpus and assigns each UNIQUE word
    an integer id.

    Returns:
        word2idx: dict mapping word -> integer id   (e.g. "cat" -> 3)
        idx2word: list mapping integer id -> word    (inverse of word2idx)
    """
    vocabulary = set()
    for sentence in corpus:
        for word in sentence:
            vocabulary.add(word)

    # sorted() makes the id assignment deterministic/reproducible across runs
    idx2word = sorted(vocabulary)
    word2idx = {word: idx for idx, word in enumerate(idx2word)}
    return word2idx, idx2word

## Step 4: Generate training pairs (center word, context word)

Skip-gram training pairs are built by sliding a window over each sentence. For every word `w` at position `i`, every OTHER word that falls within `window_size` positions to the left or right is treated as a "context word" that `w` should be able to predict.

**Example:** `sentence = ["the", "cat", "sat", "on", "the", "mat"]`, `c = 2`

Center word `"sat"` (position `i=2`) pairs with: `"the"` (i-2), `"cat"` (i-1), `"on"` (i+1), `"the"` (i+2)

→ pairs: `(sat, the), (sat, cat), (sat, on), (sat, the)`

In [3]:
def generate_training_pairs(corpus, word2idx, window_size):
    """
    Returns:
        pairs: list of (center_id, context_id) integer tuples
    """
    pairs = []
    for sentence in corpus:
        sentence_len = len(sentence)
        for i, w in enumerate(sentence):
            center_id = word2idx[w]

            # j ranges over every position within the window around i,
            # clipped to the sentence boundaries so we never index out of range
            window_start = max(0, i - window_size)
            window_end = min(sentence_len, i + window_size + 1)  # +1: range() end is exclusive

            for j in range(window_start, window_end):
                if i != j:  # "If i != j" from the pseudocode: skip the center word itself
                    context_word = sentence[j]
                    context_id = word2idx[context_word]
                    pairs.append((center_id, context_id))
    return pairs

## Softmax

Turns raw scores into a probability distribution over the vocabulary:

$$\text{probabilities}[k] = \frac{e^{\text{scores}[k]}}{\sum_j e^{\text{scores}[j]}}$$

We subtract `max(scores)` before exponentiating purely for **numerical stability** (prevents `exp()` from overflowing for large scores). This is mathematically a no-op — it does not change the resulting probabilities, because the max cancels out in the ratio of exponentials.

In [4]:
def softmax(scores):
    shifted = scores - np.max(scores)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores)

## Steps 2, 3, 5, 6: the main training routine

Trains Skip-gram word2vec embeddings from scratch using plain gradient descent (one training pair processed at a time, i.e. SGD with batch size = 1), exactly as described in the pseudocode.

**Backpropagation note:** for softmax + cross-entropy loss, the gradient of the loss w.r.t. the raw `scores` simplifies beautifully to:

`error = probabilities - one_hot(context)`

i.e. the whole probability vector, except we subtract 1 at the position of the true context word. This is exactly what `error[context] -= 1` computes.

In [5]:
def word2vec_skipgram(corpus, c, d, eta, T, seed=42, verbose=True):
    """
    Parameters:
        corpus : list[list[str]] - tokenized sentences
        c      : int   - context window size
        d      : int   - embedding dimensionality
        eta    : float - learning rate
        T      : int   - number of epochs (full passes over all pairs)
        seed   : int   - RNG seed, so results are reproducible
        verbose: bool  - print average loss per epoch

    Returns:
        E        : np.ndarray of shape (V, d) - the trained word embeddings
        word2idx : dict mapping word -> row index in E
        idx2word : list mapping row index -> word
    """
    rng = np.random.default_rng(seed)

    # ---- 1. Build vocabulary V from the corpus -----------------------
    word2idx, idx2word = build_vocabulary(corpus)
    V = len(idx2word)

    # ---- 2. Initialize input embeddings E randomly --------------------
    # E has one row per vocabulary word; E[word_id] is that word's embedding.
    # Small random values (scaled by 0.01) keep initial scores/gradients small,
    # which helps training start smoothly instead of saturating softmax.
    E = rng.normal(loc=0.0, scale=0.01, size=(V, d))

    # ---- 3. Initialize output weights W randomly -----------------------
    # W maps a center word's embedding (size d) to a score for every
    # vocabulary word (size V), so W has shape (d, V).
    W = rng.normal(loc=0.0, scale=0.01, size=(d, V))

    # ---- 4. Generate training pairs ------------------------------------
    pairs = generate_training_pairs(corpus, word2idx, window_size=c)

    # ---- 5. Training loop ------------------------------------------------
    for epoch in range(1, T + 1):
        # "Shuffle training pairs": prevents the network from learning
        # spurious patterns tied to sentence order, and improves SGD convergence.
        shuffled_pairs = pairs.copy()
        rng.shuffle(shuffled_pairs)

        total_loss = 0.0

        for center, context in shuffled_pairs:
            # ----------------- Forward pass -----------------
            h = E[center]                 # h: (d,)   the center word's current embedding
            scores = h @ W                # scores: (V,)  similarity of h to every word's output vector
            probabilities = softmax(scores)  # probabilities: (V,) distribution over the vocabulary

            # ----------------- Compute loss ------------------
            # Cross-entropy loss: how "surprised" the model is that the true
            # context word got probability probabilities[context].
            # Add a tiny epsilon to avoid log(0) if a probability underflows to 0.
            loss = -np.log(probabilities[context] + 1e-10)
            total_loss += loss

            # ----------------- Backpropagation ----------------
            # error = probabilities - one_hot(context)
            error = probabilities.copy()  # error: (V,)
            error[context] -= 1

            # d(loss)/dW = outer product of h and error -> shape (d, V), matches W
            gradient_W = np.outer(h, error)

            # d(loss)/dh = W applied to error -> shape (d,), matches h / E[center]
            gradient_h = W @ error

            # ----------------- Update parameters ----------------
            W = W - eta * gradient_W
            E[center] = E[center] - eta * gradient_h

        if verbose:
            avg_loss = total_loss / len(shuffled_pairs)
            print(f"Epoch {epoch:3d}/{T} | avg loss: {avg_loss:.4f}")

    # ---- 6. Return E ------------------------------------------------------
    return E, word2idx, idx2word

## Helper: inspect the learned embeddings via cosine similarity

This is the standard way to sanity-check that word2vec has learned something meaningful: words used in similar contexts should end up with similar vectors (high cosine similarity).

In [6]:
def most_similar(word, E, word2idx, idx2word, top_n=5):
    """
    Finds the `top_n` words whose embeddings are closest (by cosine
    similarity) to the embedding of `word`.
    """
    if word not in word2idx:
        raise ValueError(f"'{word}' is not in the vocabulary.")

    word_vec = E[word2idx[word]]
    word_norm = np.linalg.norm(word_vec) + 1e-10

    similarities = []
    for other_word, idx in word2idx.items():
        if other_word == word:
            continue
        other_vec = E[idx]
        other_norm = np.linalg.norm(other_vec) + 1e-10
        cosine_sim = np.dot(word_vec, other_vec) / (word_norm * other_norm)
        similarities.append((other_word, cosine_sim))

    similarities.sort(key=lambda pair: pair[1], reverse=True)
    return similarities[:top_n]

## Demo / example usage

A tiny toy corpus: each sentence is a list of already-tokenized words. In a real project you would lowercase, strip punctuation, and tokenize raw text to produce this structure.

In [7]:
corpus = [
    ["the", "king", "rules", "the", "kingdom"],
    ["the", "queen", "rules", "the", "kingdom"],
    ["the", "king", "and", "the", "queen", "love", "each", "other"],
    ["the", "man", "walks", "the", "dog"],
    ["the", "woman", "walks", "the", "dog"],
    ["the", "man", "and", "the", "woman", "love", "each", "other"],
    ["paris", "is", "the", "capital", "of", "france"],
    ["berlin", "is", "the", "capital", "of", "germany"],
    ["the", "king", "lives", "in", "paris"],
    ["the", "queen", "lives", "in", "berlin"],
]

# Hyperparameters, named exactly as in the pseudocode:
c = 2       # window size
d = 10      # embedding dimension
eta = 0.05  # learning rate
T = 200     # epochs

### Preview the generated training pairs

Before training, let's build the vocabulary and run `generate_training_pairs()` on its own so we can print the `(center, context)` word pairs in human-readable form and see exactly what step 4 of the pseudocode produces. `word2vec_skipgram()` below regenerates these internally when training runs.

In [8]:
_word2idx, _idx2word = build_vocabulary(corpus)
_pairs = generate_training_pairs(corpus, _word2idx, window_size=c)

print(f"Generated {len(_pairs)} training pairs (window size c={c}).")
print("First 20 pairs (center, context):")
for center_id, context_id in _pairs[:20]:
    print(f"  ({_idx2word[center_id]}, {_idx2word[context_id]})")
print("...")

Generated 172 training pairs (window size c=2).
First 20 pairs (center, context):
  (the, king)
  (the, rules)
  (king, the)
  (king, rules)
  (king, the)
  (rules, the)
  (rules, king)
  (rules, the)
  (rules, kingdom)
  (the, king)
  (the, rules)
  (the, kingdom)
  (kingdom, rules)
  (kingdom, the)
  (the, queen)
  (the, rules)
  (queen, the)
  (queen, rules)
  (queen, the)
  (rules, the)
...


In [9]:
E, word2idx, idx2word = word2vec_skipgram(corpus, c=c, d=d, eta=eta, T=T)

Epoch   1/200 | avg loss: 3.0910
Epoch   2/200 | avg loss: 3.0908
Epoch   3/200 | avg loss: 3.0904
Epoch   4/200 | avg loss: 3.0894
Epoch   5/200 | avg loss: 3.0868
Epoch   6/200 | avg loss: 3.0796
Epoch   7/200 | avg loss: 3.0599
Epoch   8/200 | avg loss: 3.0083
Epoch   9/200 | avg loss: 2.8951
Epoch  10/200 | avg loss: 2.7632
Epoch  11/200 | avg loss: 2.6905
Epoch  12/200 | avg loss: 2.6685
Epoch  13/200 | avg loss: 2.6448
Epoch  14/200 | avg loss: 2.6228
Epoch  15/200 | avg loss: 2.6007
Epoch  16/200 | avg loss: 2.5867
Epoch  17/200 | avg loss: 2.5654
Epoch  18/200 | avg loss: 2.5497
Epoch  19/200 | avg loss: 2.5289
Epoch  20/200 | avg loss: 2.5063
Epoch  21/200 | avg loss: 2.4796
Epoch  22/200 | avg loss: 2.4494
Epoch  23/200 | avg loss: 2.4216
Epoch  24/200 | avg loss: 2.3859
Epoch  25/200 | avg loss: 2.3530
Epoch  26/200 | avg loss: 2.3211


Epoch  27/200 | avg loss: 2.2831
Epoch  28/200 | avg loss: 2.2467
Epoch  29/200 | avg loss: 2.2095
Epoch  30/200 | avg loss: 2.1668
Epoch  31/200 | avg loss: 2.1330
Epoch  32/200 | avg loss: 2.1054
Epoch  33/200 | avg loss: 2.0801
Epoch  34/200 | avg loss: 2.0539


Epoch  35/200 | avg loss: 2.0280
Epoch  36/200 | avg loss: 2.0163
Epoch  37/200 | avg loss: 1.9950
Epoch  38/200 | avg loss: 1.9828
Epoch  39/200 | avg loss: 1.9584
Epoch  40/200 | avg loss: 1.9581
Epoch  41/200 | avg loss: 1.9595
Epoch  42/200 | avg loss: 1.9431
Epoch  43/200 | avg loss: 1.9440
Epoch  44/200 | avg loss: 1.9441
Epoch  45/200 | avg loss: 1.9353
Epoch  46/200 | avg loss: 1.9287
Epoch  47/200 | avg loss: 1.9264
Epoch  48/200 | avg loss: 1.9320
Epoch  49/200 | avg loss: 1.9301
Epoch  50/200 | avg loss: 1.9203
Epoch  51/200 | avg loss: 1.9275
Epoch  52/200 | avg loss: 1.9127
Epoch  53/200 | avg loss: 1.9215
Epoch  54/200 | avg loss: 1.9143
Epoch  55/200 | avg loss: 1.9242
Epoch  56/200 | avg loss: 1.9237
Epoch  57/200 | avg loss: 1.9144
Epoch  58/200 | avg loss: 1.9118
Epoch  59/200 | avg loss: 1.9136
Epoch  60/200 | avg loss: 1.9037
Epoch  61/200 | avg loss: 1.9157
Epoch  62/200 | avg loss: 1.9097
Epoch  63/200 | avg loss: 1.8999
Epoch  64/200 | avg loss: 1.9129
Epoch  65/

Epoch  68/200 | avg loss: 1.8945
Epoch  69/200 | avg loss: 1.9131
Epoch  70/200 | avg loss: 1.8971
Epoch  71/200 | avg loss: 1.9033
Epoch  72/200 | avg loss: 1.9176
Epoch  73/200 | avg loss: 1.9146
Epoch  74/200 | avg loss: 1.9091
Epoch  75/200 | avg loss: 1.8870
Epoch  76/200 | avg loss: 1.9155


Epoch  77/200 | avg loss: 1.8983
Epoch  78/200 | avg loss: 1.9097
Epoch  79/200 | avg loss: 1.9099
Epoch  80/200 | avg loss: 1.8957
Epoch  81/200 | avg loss: 1.9160
Epoch  82/200 | avg loss: 1.9209
Epoch  83/200 | avg loss: 1.9155
Epoch  84/200 | avg loss: 1.9084
Epoch  85/200 | avg loss: 1.9068
Epoch  86/200 | avg loss: 1.9116
Epoch  87/200 | avg loss: 1.8977
Epoch  88/200 | avg loss: 1.9095
Epoch  89/200 | avg loss: 1.9072
Epoch  90/200 | avg loss: 1.9124
Epoch  91/200 | avg loss: 1.9065
Epoch  92/200 | avg loss: 1.9111
Epoch  93/200 | avg loss: 1.9096
Epoch  94/200 | avg loss: 1.9140
Epoch  95/200 | avg loss: 1.9015
Epoch  96/200 | avg loss: 1.9154
Epoch  97/200 | avg loss: 1.9114
Epoch  98/200 | avg loss: 1.9035
Epoch  99/200 | avg loss: 1.9194
Epoch 100/200 | avg loss: 1.8911
Epoch 101/200 | avg loss: 1.9009
Epoch 102/200 | avg loss: 1.9161
Epoch 103/200 | avg loss: 1.9101
Epoch 104/200 | avg loss: 1.9086
Epoch 105/200 | avg loss: 1.9084
Epoch 106/200 | avg loss: 1.8958
Epoch 107/

Epoch 109/200 | avg loss: 1.9245
Epoch 110/200 | avg loss: 1.9010
Epoch 111/200 | avg loss: 1.8898
Epoch 112/200 | avg loss: 1.9079
Epoch 113/200 | avg loss: 1.9169
Epoch 114/200 | avg loss: 1.8992
Epoch 115/200 | avg loss: 1.9038
Epoch 116/200 | avg loss: 1.8935
Epoch 117/200 | avg loss: 1.8946
Epoch 118/200 | avg loss: 1.9176


Epoch 119/200 | avg loss: 1.9054
Epoch 120/200 | avg loss: 1.9105
Epoch 121/200 | avg loss: 1.8948
Epoch 122/200 | avg loss: 1.8989
Epoch 123/200 | avg loss: 1.9084
Epoch 124/200 | avg loss: 1.9114
Epoch 125/200 | avg loss: 1.8970
Epoch 126/200 | avg loss: 1.8828
Epoch 127/200 | avg loss: 1.9067
Epoch 128/200 | avg loss: 1.9053
Epoch 129/200 | avg loss: 1.9064
Epoch 130/200 | avg loss: 1.9109
Epoch 131/200 | avg loss: 1.8951
Epoch 132/200 | avg loss: 1.9006
Epoch 133/200 | avg loss: 1.9023
Epoch 134/200 | avg loss: 1.8989
Epoch 135/200 | avg loss: 1.8972
Epoch 136/200 | avg loss: 1.9047
Epoch 137/200 | avg loss: 1.9132
Epoch 138/200 | avg loss: 1.9098
Epoch 139/200 | avg loss: 1.8863
Epoch 140/200 | avg loss: 1.9089
Epoch 141/200 | avg loss: 1.8671
Epoch 142/200 | avg loss: 1.9182
Epoch 143/200 | avg loss: 1.9076
Epoch 144/200 | avg loss: 1.8974
Epoch 145/200 | avg loss: 1.9071
Epoch 146/200 | avg loss: 1.8903
Epoch 147/200 | avg loss: 1.9164
Epoch 148/200 | avg loss: 1.8806
Epoch 149/

Epoch 152/200 | avg loss: 1.8976
Epoch 153/200 | avg loss: 1.8956
Epoch 154/200 | avg loss: 1.9093
Epoch 155/200 | avg loss: 1.9082
Epoch 156/200 | avg loss: 1.8924
Epoch 157/200 | avg loss: 1.9079
Epoch 158/200 | avg loss: 1.9100
Epoch 159/200 | avg loss: 1.8972
Epoch 160/200 | avg loss: 1.8990
Epoch 161/200 | avg loss: 1.9007


Epoch 162/200 | avg loss: 1.9002
Epoch 163/200 | avg loss: 1.9042
Epoch 164/200 | avg loss: 1.8916
Epoch 165/200 | avg loss: 1.8979
Epoch 166/200 | avg loss: 1.8958
Epoch 167/200 | avg loss: 1.9028
Epoch 168/200 | avg loss: 1.9003
Epoch 169/200 | avg loss: 1.8990
Epoch 170/200 | avg loss: 1.9037
Epoch 171/200 | avg loss: 1.9003
Epoch 172/200 | avg loss: 1.9007
Epoch 173/200 | avg loss: 1.9000
Epoch 174/200 | avg loss: 1.9032
Epoch 175/200 | avg loss: 1.9003
Epoch 176/200 | avg loss: 1.9033
Epoch 177/200 | avg loss: 1.8877
Epoch 178/200 | avg loss: 1.9077
Epoch 179/200 | avg loss: 1.8941
Epoch 180/200 | avg loss: 1.9035
Epoch 181/200 | avg loss: 1.8909
Epoch 182/200 | avg loss: 1.8838
Epoch 183/200 | avg loss: 1.9100
Epoch 184/200 | avg loss: 1.9067
Epoch 185/200 | avg loss: 1.9103
Epoch 186/200 | avg loss: 1.8992
Epoch 187/200 | avg loss: 1.9060
Epoch 188/200 | avg loss: 1.8954
Epoch 189/200 | avg loss: 1.9081
Epoch 190/200 | avg loss: 1.8958
Epoch 191/200 | avg loss: 1.9021
Epoch 192/

Epoch 193/200 | avg loss: 1.8865
Epoch 194/200 | avg loss: 1.9042
Epoch 195/200 | avg loss: 1.9014
Epoch 196/200 | avg loss: 1.8955
Epoch 197/200 | avg loss: 1.8973
Epoch 198/200 | avg loss: 1.9059
Epoch 199/200 | avg loss: 1.8939
Epoch 200/200 | avg loss: 1.8936


In [10]:
print("Vocabulary size:", len(idx2word))
print("Embedding matrix shape:", E.shape)

for query_word in ["king", "queen", "paris"]:
    print(f"\nWords most similar to '{query_word}':")
    for word, score in most_similar(query_word, E, word2idx, idx2word, top_n=5):
        print(f"  {word:10s}  cosine_sim = {score:.4f}")

Vocabulary size: 22
Embedding matrix shape: (22, 10)

Words most similar to 'king':
  queen       cosine_sim = 0.8791
  kingdom     cosine_sim = 0.7864
  man         cosine_sim = 0.6907
  paris       cosine_sim = 0.6833
  berlin      cosine_sim = 0.6661

Words most similar to 'queen':
  king        cosine_sim = 0.8791
  woman       cosine_sim = 0.6989
  kingdom     cosine_sim = 0.6694
  man         cosine_sim = 0.5666
  paris       cosine_sim = 0.5358

Words most similar to 'paris':
  berlin      cosine_sim = 0.9995
  king        cosine_sim = 0.6833
  capital     cosine_sim = 0.6265
  queen       cosine_sim = 0.5358
  kingdom     cosine_sim = 0.4062


### What to expect

Even with this tiny toy corpus and only 200 epochs, words that appear in similar contexts (e.g. `king`/`queen`, `paris`/`berlin`) should cluster together with relatively high cosine similarity, because Skip-gram pushes a word's embedding to be predictive of the words that surround it — words sharing contexts end up with similar vectors.